# Fruit Quality, Ripeness & Defect Assessment — CNN Classifier

**Task**: classify a photo of a single fruit into one of 12 classes (6 fruit types × good/bad quality).

**Data**: `Fruit Quality Classification/` — a Roboflow YOLO-segmentation export (Kaggle `abrars2/fruit-quality-classificaltion-and-detection`). We only use the class label of the first annotated object per image; the polygon geometry is ignored since this is a whole-image classification task, not detection/segmentation.

**Model**: MobileNetV2 transfer learning — small and fast enough to train on a CPU or low-end GPU.

**Notes on the raw data** (see the planning discussion for how these were found):
- The dataset ships 14 raw classes, but `Good_Lime` and `Good_Pomegranate` were each accidentally split into two near-duplicate classes by an inconsistent-hyphenation typo. We merge them back down to the real 12 classes below.
- ~15% of images have more than one fruit in frame; we keep only the first labeled object per image for simplicity.
- ~213 label files have no matching image in the downloaded zip (a packaging gap) and are skipped automatically.

In [ ]:
# If needed: !pip install tensorflow pandas numpy scikit-learn matplotlib seaborn pillow

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 1. Build a label index from the YOLO label files

Each `.txt` label file has one line per fruit in the photo: `class_id  <polygon points...>`. We only need `class_id`, and we only take the **first** line (first fruit) per image.

In [ ]:
DATASET_DIR = "Fruit Quality Classification"
SPLITS = ["train", "valid"]  # pool both source folders; we make our own split later

RAW_CLASS_NAMES = [
    "Bad_Apple 0-1 day", "Bad_Banana-0 day", "Bad_Guava 0 day", "Bad_Lime-0-1-day",
    "Bad_Orange-0-1 day", "Bad_Pomegranate-0-1-day", "Good_Apple 10-21 days",
    "Good_Banana-2-3 days", "Good_Guava 5-7 days", "Good_Lime-10-21 days",
    "Good_Lime-10-21-days", "Good_Orange-15-21 days", "Good_Pomegranate 50-60-days",
    "Good_Pomegranate-50-60-days",
]

# Raw id -> merged id. Ids 9/10 (Good_Lime) and 12/13 (Good_Pomegranate) are the
# same real-world class, split apart by an inconsistent-hyphenation typo upstream.
RAW_TO_MERGED = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5,
                 6: 6, 7: 7, 8: 8,
                 9: 9, 10: 9,
                 11: 10,
                 12: 11, 13: 11}

CLASS_NAMES = ["Bad_Apple", "Bad_Banana", "Bad_Guava", "Bad_Lime", "Bad_Orange", "Bad_Pomegranate",
               "Good_Apple", "Good_Banana", "Good_Guava", "Good_Lime", "Good_Orange", "Good_Pomegranate"]
NUM_CLASSES = len(CLASS_NAMES)


def build_label_index(dataset_dir, splits):
    """Return one (image_path, merged_label) row per usable image."""
    rows = []
    skipped_empty = 0
    skipped_orphan = 0
    for split in splits:
        images_dir = os.path.join(dataset_dir, split, "images")
        labels_dir = os.path.join(dataset_dir, split, "labels")
        for fname in os.listdir(labels_dir):
            if not fname.endswith(".txt"):
                continue
            with open(os.path.join(labels_dir, fname), "r") as f:
                first_line = f.readline().strip()
            if not first_line:
                skipped_empty += 1  # unlabeled image, nothing to classify
                continue
            raw_id = int(first_line.split()[0])
            image_path = os.path.join(images_dir, fname[:-4] + ".jpg")
            if not os.path.exists(image_path):
                skipped_orphan += 1  # label with no matching image in the zip
                continue
            rows.append((image_path, RAW_TO_MERGED[raw_id]))
    df = pd.DataFrame(rows, columns=["image_path", "label"])
    print(f"Loaded {len(df)} images  |  skipped {skipped_empty} empty labels, {skipped_orphan} orphaned labels")
    return df


data_df = build_label_index(DATASET_DIR, SPLITS)
data_df["class_name"] = data_df["label"].map(lambda i: CLASS_NAMES[i])
data_df["class_name"].value_counts()

## 2. Train / validation / test split

We pool the source `train` and `valid` folders and make our own stratified 70/15/15 split, rather than trusting the original (small) `valid` folder alone for evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data_df, test_size=0.30, stratify=data_df["label"], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

## 3. A quick look at the data

In [ ]:
counts = data_df["class_name"].value_counts().reindex(CLASS_NAMES)
plt.figure(figsize=(10, 4))
plt.bar(counts.index, counts.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("images")
plt.title("Class distribution (full pooled dataset)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, class_id in zip(axes.flat, range(NUM_CLASSES)):
    sample_path = data_df[data_df["label"] == class_id]["image_path"].iloc[0]
    ax.imshow(Image.open(sample_path))
    ax.set_title(CLASS_NAMES[class_id], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. A lightweight `tf.data` pipeline

Small image size, cached decoded images (the whole dataset is only ~1,900 images so this comfortably fits in RAM), and light augmentation on the training split only. Drop `.cache()` below if your machine is RAM-constrained.

In [ ]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label


data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


def make_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["image_path"].values, df["label"].values))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.keras.applications.mobilenet_v2.preprocess_input(x), y),
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

## 5. Model — MobileNetV2 transfer learning

Frozen pretrained base + a small classification head. Cheap to train on CPU since only the head is trainable in phase 1.

In [ ]:
def build_model(num_classes, img_size):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=img_size + (3,), include_top=False, weights="imagenet")
    base_model.trainable = False  # frozen for phase 1

    inputs = tf.keras.Input(shape=img_size + (3,))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs), base_model


model, base_model = build_model(NUM_CLASSES, IMG_SIZE)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
# Classes are imbalanced (e.g. Bad_Orange vs Good_Apple) - weight the loss to compensate.
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
class_weight_dict = dict(enumerate(class_weights))

## 6. Train (phase 1: frozen base)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", save_best_only=True, monitor="val_accuracy"),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=15,
                     class_weight=class_weight_dict, callbacks=callbacks)

In [ ]:
def plot_history(hist, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist.history["accuracy"], label="train")
    axes[0].plot(hist.history["val_accuracy"], label="val")
    axes[0].set_title(f"{title} - Accuracy"); axes[0].legend()
    axes[1].plot(hist.history["loss"], label="train")
    axes[1].plot(hist.history["val_loss"], label="val")
    axes[1].set_title(f"{title} - Loss"); axes[1].legend()
    plt.tight_layout()
    plt.show()


plot_history(history, "Phase 1 (frozen base)")

## 7. Optional fine-tuning

Unfreezes the last 30 layers of MobileNetV2 and trains a bit more at a low learning rate, usually gives a small accuracy bump. **Skip this cell** (set `DO_FINE_TUNE = False`) if it's too slow on your machine — phase 1 alone already gives a working model.

In [ ]:
DO_FINE_TUNE = True

if DO_FINE_TUNE:
    base_model.trainable = True
    fine_tune_at = len(base_model.layers) - 30
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5,
                              class_weight=class_weight_dict, callbacks=callbacks)
    plot_history(history_fine, "Phase 2 (fine-tuned)")

## 8. Evaluate on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = test_df["label"].values
y_pred = np.argmax(model.predict(test_ds), axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 9. Try it on a single image

In [ ]:
def predict_single_image(path, model):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img = tf.expand_dims(img, axis=0)

    probs = model.predict(img, verbose=0)[0]
    pred_id = int(np.argmax(probs))

    plt.imshow(Image.open(path))
    plt.axis("off")
    plt.title(f"Predicted: {CLASS_NAMES[pred_id]} ({probs[pred_id]:.1%})")
    plt.show()


sample_path = test_df.sample(1, random_state=1)["image_path"].iloc[0]
predict_single_image(sample_path, model)